In [3]:
# !pip install requests

In [4]:
import requests

In [5]:
r = requests.post("http://localhost:11434/api/embeddings", json={
    "model": "bge-m3",
    "prompt": "Yash is a good boy"
})

In [6]:
# print(r.json())

In [7]:
embedding = r.json()["embedding"]

In [8]:
print(embedding[0:5])

[-0.3663097321987152, -0.4248668849468231, -0.4697731137275696, -0.2662394046783447, -0.6197561025619507]


## Embeddings

In [9]:
def create_embedding(text_list):
    r = requests.post("http://localhost:11434/api/embed", json={
        "model": "bge-m3",
        "input": text_list
    })

    embedding = r.json()["embeddings"]
    return embedding

In [10]:
# print(create_embedding(["My name is", "Yashpal"]))

## Read Chunks and Create Embeddings

In [11]:
import json
import os

In [12]:
jsons = os.listdir("../data/jsons")
print(jsons)

['100_Exercise 17 - Design This Layout .mp3.json', '33_Exercise 4 - Multi Color Website .mp3.json', '41_Exercise 6 - Navbar using Flexbox .mp3.json', '65_JavaScript Exercise 11 - Calculate the Factorial .mp3.json', '70_JavaScript Exercise 12 - Color the Boxes .mp3.json']


In [13]:
my_dict = []
chunk_id = 0

In [14]:
for json_file in jsons:
    with open(f"../data/jsons/{json_file}") as f:
        content = json.load(f)
    embeddings = create_embedding([c["text"] for c in content["chunks"]])
    
    for i, chunk in enumerate(content["chunks"]):
        # print(chunk)
        chunk["chunk_id"] = chunk_id
        chunk["embedding"] = embeddings[i]
        chunk_id += 1
        my_dict.append(chunk)

In [15]:
print("Total chunks:", len(my_dict))
print("First chunk:", my_dict[0]["text"])
print("Embedding dimensions:", len(my_dict[0]["embedding"]))

Total chunks: 312
First chunk:  Guys, I told you about Tailwind CSS.
Embedding dimensions: 1024


## Saving into DataFrame

In [16]:
import pandas as pd

In [17]:
df = pd.DataFrame.from_records(my_dict)
df

,number,title,start,end,text,chunk_id,embedding
0,100,Exercise 17 - Design This Layout,0.00,2.50,"Guys, I told you about Tailwind CSS.",0,"[-0.05321666, 0.002405325, -0.050752055, 0.010..."
1,100,Exercise 17 - Design This Layout,2.50,6.38,"In this video, you have to use it and design ...",1,"[-0.021918219, 0.028098807, -0.054138362, 0.01..."
2,100,Exercise 17 - Design This Layout,6.38,7.92,so that your confidence increases.,2,"[-0.039596494, 0.027565543, -0.07415438, 0.001..."
3,100,Exercise 17 - Design This Layout,7.92,11.76,Like this video and write Sigma Patch Over in...,3,"[-0.05712266, -0.0067729456, -0.016123166, 0.0..."
4,100,Exercise 17 - Design This Layout,11.76,13.38,Let's go to the computer screen.,4,"[-0.010954705, 0.02127551, -0.07029707, -0.010..."
...,...,...,...,...,...,...,...
307,70,JavaScript Exercise 12 - Color the Boxes,204.00,206.00,"If you have not accessed the playlist, then d...",307,"[-0.019823343, 0.029636787, -0.031437244, 0.01..."
308,70,JavaScript Exercise 12 - Color the Boxes,206.00,207.00,That's all for now guys,308,"[-0.015002052, 0.013911063, -0.057893075, -0.0..."
309,70,JavaScript Exercise 12 - Color the Boxes,207.00,209.00,Thank you so much guys for watching this video,309,"[-0.0386146, -0.0071998667, -0.047328908, -0.0..."
310,70,JavaScript Exercise 12 - Color the Boxes,209.00,212.00,And I will see you next time,310,"[-0.0033871173, 0.040171187, -0.033758752, 0.0..."


## Find similarities of query with embeddings

In [25]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [34]:
incoming_query = input("Ask a Question: ")
question_emb = create_embedding([incoming_query])[0]
# print(question_emb)

similarities = cosine_similarity(np.vstack(df["embedding"]), [question_emb]).flatten()

top_result = 3
max_index = similarities.argsort()[::-1][0:top_result]
print(max_index)
new_df = df.loc[max_index]
new_df

Ask a Question:  tailwind css


[ 0 23 14]


,number,title,start,end,text,chunk_id,embedding
0,100,Exercise 17 - Design This Layout,0.00,2.50,"Guys, I told you about Tailwind CSS.",0,"[-0.05321666, 0.002405325, -0.050752055, 0.010..."
23,100,Exercise 17 - Design This Layout,77.80,80.46,But you also have to do it with the help of T...,23,"[-0.062160436, 0.032005697, -0.026386674, 0.00..."
14,100,Exercise 17 - Design This Layout,52.38,56.04,You have to use Tailwind CSS to design a layout.,14,"[-0.058305018, 0.008270422, -0.03620454, -0.00..."
